# Teller API Queries

This notebook demonstrates how to use the `TellerClient` to query your bank account balances and transactions.

In [2]:
import pandas as pd
import sys
import os

# Add backend to path to allow imports
sys.path.append(os.path.abspath('../backend'))

from integrations.teller.client import TellerClient

# Display settings for Pandas
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## Configuration

Set your certificate paths and access token here. 
**Note:** Keep these values secure.

In [3]:
# TODO: Replace with your actual paths and token
CERT_PATH = '/Users/alan/richtato/teller/certificate.pem'
KEY_PATH = '/Users/alan/richtato/teller/private_key.pem'
ACCESS_TOKEN = 'app_plltj26cnov2mv4vce000'

## Initialize Client

In [4]:
client = TellerClient(cert_path=CERT_PATH, key_path=KEY_PATH, access_token=ACCESS_TOKEN)

## Get Accounts & Balances

In [5]:
try:
    accounts = client.get_accounts()
    df_accounts = pd.DataFrame(accounts)
    
    # Display relevant columns if available
    cols = ['id', 'name', 'type', 'currency', 'last_four']
    # Extract balance to a separate column for cleaner display if it exists
    if 'balances' in df_accounts.columns:
        df_accounts['available_balance'] = df_accounts['balances'].apply(lambda x: x.get('available') if isinstance(x, dict) else None)
        df_accounts['ledger_balance'] = df_accounts['balances'].apply(lambda x: x.get('ledger') if isinstance(x, dict) else None)
        cols.extend(['available_balance', 'ledger_balance'])
        
    print("Accounts:")
    display(df_accounts[cols] if set(cols).issubset(df_accounts.columns) else df_accounts)
    
    # Save first account ID for transaction queries
    if accounts:
        account_id = accounts[0]['id']
        print(f"\nUsing Account ID for transactions: {account_id}")
    else:
        print("No accounts found.")
        account_id = None
except Exception as e:
    print(f"Error fetching accounts: {e}")

Error fetching accounts: 403 Client Error: Forbidden for url: https://api.teller.io/accounts


## Get Transactions

Fetch latest transactions with optional count limit.

In [ ]:
if account_id:
    try:
        # Fetch latest 50 transactions
        transactions = client.get_transactions(account_id, count=50)
        df_transactions = pd.DataFrame(transactions)
        
        print(f"Fetched {len(transactions)} transactions.")
        if not df_transactions.empty:
            display(df_transactions[['date', 'description', 'amount', 'status', 'id']].head())
    except Exception as e:
        print(f"Error fetching transactions: {e}")

## Filter Transactions by Date

Filter the fetched transactions locally by date range.

In [ ]:
if account_id and 'transactions' in locals() and transactions:
    # Define date range
    START_DATE = '2023-01-01'
    END_DATE = '2023-12-31'
    
    filtered_txns = client.filter_transactions(transactions, start_date=START_DATE, end_date=END_DATE)
    df_filtered = pd.DataFrame(filtered_txns)
    
    print(f"Transactions between {START_DATE} and {END_DATE}: {len(filtered_txns)}")
    if not df_filtered.empty:
        display(df_filtered[['date', 'description', 'amount', 'status']])
    else:
        print("No transactions found in this date range.")